# KLA Image Restoration — Quickstart Notebook

Run the cells **in order, top to bottom**. Read the comment at the top of each
cell before you run it — a few need you to edit one line.

Works on Kaggle or Google Colab. Make sure the GPU is turned on first:
- **Kaggle:** right sidebar -> Accelerator -> GPU T4 x2
- **Colab:** Runtime -> Change runtime type -> T4 GPU


## 1. Check you actually have a GPU

If this errors or shows nothing, your GPU is not enabled. Go back and turn it on.

In [ ]:
!nvidia-smi

### What GPU do you have?

Note the compute capability. 7.5 is a T4: it has fp16 tensor cores but **no**
bf16 tensor cores, so bf16 runs unaccelerated.

Do not pick `--amp` from this cell. It gets decided by measurement in step 6b,
because a dtype that is faster but silently produces NaN gradients is worse
than useless.


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"compute capability: {cap[0]}.{cap[1]}")
    print("  fp16 tensor cores:", cap >= (7, 0))
    print("  bf16 tensor cores:", cap >= (8, 0))
else:
    print("NO GPU. Stop here and enable the accelerator.")

## 2. Install dependencies

In [ ]:
!pip install -q tifffile lpips

## 3. Get the code

If you uploaded this folder as a **Kaggle dataset**, copy it into
`/kaggle/working/` first. Running straight from `/kaggle/input/` fails: that
path is read-only and the scripts need to write `manifest.json` and checkpoints.


In [ ]:
import os, shutil

# EDIT this to your dataset name, or comment the block out if you uploaded
# the folder directly into /kaggle/working/.
SRC = "/kaggle/input/kla-restoration/kla_restoration"
PROJECT = "/kaggle/working/kla_restoration"

if os.path.exists(SRC) and not os.path.exists(PROJECT):
    shutil.copytree(SRC, PROJECT)

os.chdir(PROJECT)
print(os.getcwd())
print(sorted(os.listdir()))

## 4. Smoke test — verify everything works

Takes ~2 minutes. Must end with `Smoke test passed.`

**If this fails, stop and fix it before going further.** An error here has one
clear cause; the same error later will have five.

In [ ]:
!python smoke_test.py

## 5. Point at your real data

Your files must be in two folders with matching filenames:

```
data/gt/         clean, high-resolution  (the correct answers)
data/degraded/   noisy, low-resolution   (the model's input)
```

In [ ]:
GT_DIR  = "data/gt"          # <-- EDIT
LR_DIR  = "data/degraded"    # <-- EDIT

import os
print("GT files:", len(os.listdir(GT_DIR)))
print("LR files:", len(os.listdir(LR_DIR)))
print("\nFirst few GT:", sorted(os.listdir(GT_DIR))[:5])
print("First few LR:", sorted(os.listdir(LR_DIR))[:5])

### Look at your data with your own eyes

Do not skip this. If the pairs are mismatched, everything downstream produces
confident nonsense and nothing warns you. The two images in each row must show
the same structure.

In [ ]:
import matplotlib.pyplot as plt
from imageio_utils import list_images, read_gray

gt_files = list_images(GT_DIR)[:3]
lr_files = list_images(LR_DIR)[:3]

fig, axes = plt.subplots(3, 2, figsize=(9, 13))
for i, (g, l) in enumerate(zip(gt_files, lr_files)):
    hr, lr = read_gray(g), read_gray(l)
    axes[i, 0].imshow(lr, cmap="gray"); axes[i, 0].set_title(f"DEGRADED {lr.shape}\n{l.name}", fontsize=9)
    axes[i, 1].imshow(hr, cmap="gray"); axes[i, 1].set_title(f"GROUND TRUTH {hr.shape}\n{g.name}", fontsize=9)
    for a in axes[i]: a.axis("off")
plt.tight_layout(); plt.show()
print("Same structure in each row? If not, STOP and fix the filename pairing.")

## 6. Audit the data

Creates `manifest.json`. `--cluster_groups 6` derives groups from image content,
which you need when filenames are plain numbers and carry no origin label. It
gives groups named `c00`..`c05`.

Two things to check in the output:
- `All sampled pairs are exactly x2.`
- The `GROUPS` table lists several groups, not one


In [ ]:
!python audit_data.py --hr_dir {GT_DIR} --lr_dir {LR_DIR} \
    --out manifest.json --sample 200 --cluster_groups 6

## 6a. Preflight — is the synthetic degradation realistic?

`synth_ratio` decides what fraction of every batch is made by
`SyntheticDegrader`. If that degrader is harsher, softer or noisier than your
real degradation, that fraction of the training signal teaches the model to
solve a different problem, and it hedges toward smooth, bicubic-like output.
Nothing in the training log tells you this is happening.

Act on the `interpretation` block:
- *"matches well"* -> use `SYNTH = 0.35` below
- *"Too harsh"* -> paste the suggested ranges into `degradations.py`, keep 0.35.
  If you would rather not edit that file, use `SYNTH = 0.15`
- *"Too mild"* -> raise `speckle_sigma` in `degradations.py`
- `[FATAL] GT exceeds 1.0` -> stop. Re-run the audit with an explicit
  `--maxval`. Every metric is invalid until that is fixed.


In [ ]:
!python preflight.py --manifest manifest.json --n 80

## 6b. Choose --amp by measuring it

Runs real training steps in fp32 / bf16 / fp16 and reports, for each, whether
the weights **actually move** and how fast it is. fp16 on this model can produce
NaN gradients: the loss keeps printing sensible numbers while nothing learns.

Takes about three minutes. Copy the `USE: --amp X` line and the iteration count
for your budget into the next cells.

If a dtype comes back BROKEN and you want to know why:
`!python check_amp.py --manifest manifest.json --locate fp16`


In [ ]:
!python check_amp.py --manifest manifest.json --val_groups c04

## 6c. Sanity run — 25 minutes that protect 7 hours

Synthesis off, so this tests only whether the model can learn the **real**
degradation.

Val PSNR must climb at every checkpoint and clear roughly 25 dB.

If it is still stuck near the bicubic baseline with synthesis off and a sane
learning rate, **stop**. The problem is the data pairing, not the config, and
more GPU hours will not fix it.


In [ ]:
AMP = "bf16"   # <-- EDIT: from cell 6b

!python train.py --manifest manifest.json --preset medium \
    --iters 4000 --batch 16 --patch 64 \
    --lr 3e-4 --beta2 0.99 --synth_ratio 0.0 --amp {AMP} \
    --val_groups c04 --val_every 1000 --val_limit 128 \
    --workers 2 --cache --out /kaggle/working/sanity

## 7. Train

`--iters` is only a **ceiling**. `--fit_hours` measures your actual it/s over
steps 100-600 and shrinks the schedule to fit the budget, because a cosine
schedule that completes beats a longer one you kill partway, every time.

Set `HOURS` below your real limit to leave headroom.

Watch for:
- `[data]` — HR range must sit inside [0, 1]
- `[amp]` — reported at iteration 100. If gradients are non-finite the run
  aborts here instead of wasting the session
- `[fit]` — the iteration count it settled on
- `loss` trending down and `PSNR` in the VAL blocks trending up

For a run this long use **Save Version -> Save & Run All (Commit)** rather than
an interactive tab, which can drop.


In [ ]:
AMP    = "bf16"   # <-- EDIT: from cell 6b
SYNTH  = 0.35     # <-- EDIT: from cell 6a
HOURS  = 7.0      # <-- EDIT: your budget, minus headroom
OUT    = "/kaggle/working/run4"

!python train.py --manifest manifest.json --preset medium \
    --iters 250000 --fit_hours {HOURS} \
    --batch 16 --patch 64 \
    --lr 3e-4 --beta2 0.99 --warmup 2000 \
    --synth_ratio {SYNTH} --w_ssim 0.2 --amp {AMP} \
    --val_groups c04 --val_every 2500 --val_limit 128 \
    --workers 2 --cache --out {OUT}

### If your session died mid-training

Rerun the cell below with **identical flags**. The sample stream and the cosine
schedule both pick up where they stopped.


In [ ]:
!python train.py --manifest manifest.json --preset medium \
    --iters 250000 --fit_hours {HOURS} \
    --batch 16 --patch 64 \
    --lr 3e-4 --beta2 0.99 --warmup 2000 \
    --synth_ratio {SYNTH} --w_ssim 0.2 --amp {AMP} \
    --val_groups c04 --val_every 2500 --val_limit 128 \
    --workers 2 --cache --out {OUT} \
    --resume {OUT}/last.pth

## 8. Evaluate

The number that matters is **`delta vs bicubic`**. Bicubic is naive upscaling
with no AI at all.

- +3 dB or more: working well
- +1 to +3 dB: working, improvable
- under +1 dB: something is wrong

This runs the **full** held-out group, not the 128-image subset used for
checkpoint selection during training, so expect it to read a little lower.


In [ ]:
!python evaluate.py --ckpt {OUT}/best.pth --manifest manifest.json \
    --val_groups c04 --lpips

## 9. Check speed

You are scored on inference time, so measure it. Self-ensembling buys 0.1-0.3 dB
at 8x the cost; run both and decide with numbers, not taste.


In [ ]:
!python infer.py --ckpt {OUT}/best.pth --benchmark 256 --half
!python infer.py --ckpt {OUT}/best.pth --benchmark 256 --half --ensemble 8
!python evaluate.py --ckpt {OUT}/best.pth --manifest manifest.json \
    --val_groups c04 --ensemble 8

## 10. Generate predictions

Point `TEST_DIR` at the test images when they're released.

In [ ]:
TEST_DIR = "test/degraded"   # <-- EDIT
!python infer.py --ckpt {OUT}/best.pth --in_dir {TEST_DIR} --out_dir predictions --half

### Look at the results

Metrics can look fine while images have visible seams or artifacts. Your eyes
catch things PSNR doesn't.

In [ ]:
import matplotlib.pyplot as plt
from imageio_utils import list_images, read_gray

ins  = list_images(TEST_DIR)[:3]
outs = list_images("predictions")[:3]

fig, axes = plt.subplots(3, 2, figsize=(9, 13))
for i, (a, b) in enumerate(zip(ins, outs)):
    axes[i, 0].imshow(read_gray(a), cmap="gray"); axes[i, 0].set_title("INPUT", fontsize=9)
    axes[i, 1].imshow(read_gray(b), cmap="gray"); axes[i, 1].set_title("RESTORED", fontsize=9)
    for ax in axes[i]: ax.axis("off")
plt.tight_layout(); plt.show()